##导入相关包

In [ ]:
import pandas as pd
import numpy as np
import jieba  #jieba分词
import os   
from sklearn.feature_extraction.text import TfidfVectorizer #TF-IDF
from sklearn.model_selection import GridSearchCV            #网格搜索
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix,classification_report #预测效果的评估
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif']=['SimHei']  #这两行用于plt图像显示中文，否则plt无法显示中文
plt.rcParams['axes.unicode_minus']=False

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##簡轉繁

In [ ]:
!pip3 install opencc-python-reimplemented

#数据读取(簡->繁)

In [ ]:
# 使用pandas读取数据集
import pandas as pd
# train = pd.read_table('train.txt', sep='\t',header=None)  # 训练集 占RAM太多了，當TFIDF矩陣的時候：(news_tfidf.toarray())
train = pd.read_table('/content/drive/MyDrive/中文新闻文本标题分类Data/train.txt', sep='\t', header=None, nrows=20000)  # 训练集，仍占用太多，限制只讀20000筆
test = pd.read_table('/content/drive/MyDrive/中文新闻文本标题分类Data/test.txt', sep='\t',header=None, nrows=20000) # 测试集，仍占用太多，限制只讀20000筆

In [ ]:
# 添加列名便于对数据进行更好处理
train.columns = ["text_a",'label']
# dev.columns = ["text_a",'label']
test.columns = ["text_a"]

In [ ]:
# 定义要进行分类的14个类别
label_list=list(train.label.unique())
print(label_list)
print(len(label_list))  # 一定要14个类别

In [ ]:
# 查看训练集
train

In [ ]:
test

##遍歷 train 變數的每一行，然後將每一行的簡體字轉換為繁體字

In [ ]:
from opencc import OpenCC

cc = OpenCC('s2tw')
for index, row in train.iterrows():
  train.loc[index, 'label'] = cc.convert(str(row['label']))
  train.loc[index, 'text_a'] = cc.convert(row['text_a'])
for index, row in test.iterrows():
  #test.loc[index, 'label'] = cc.convert(str(row['label']))
  test.loc[index, 'text_a'] = cc.convert(row['text_a'])

In [ ]:
# 復原train_tmp用，便實驗
train_tmp=train.copy
# 查看训练集
train

##统计

In [ ]:
# 总类别标签分布统计
train['label'].value_counts()

In [ ]:
# 文本长度统计分析,通过分析可以看出文本较短，最长为48
train['text_a'].map(len).describe()

In [ ]:
#待研究!
# 全局设置解决matplotlib中文显示错误问题，参考：https://aistudio.baidu.com/aistudio/projectdetail/1658980
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

import matplotlib.font_manager as font_manager

# 设置显示中文
matplotlib.rcParams['font.sans-serif'] = ['FZSongYi-Z13S'] # 指定默认字体
matplotlib.rcParams['axes.unicode_minus'] = False # 解决保存图像是负号'-'显示为方块的问题
# 设置字体大小
matplotlib.rcParams['font.size'] = 16

# 可视化类别标签分布情况
train['label'].value_counts(normalize=True).plot(kind='bar');

#X
#尚無停用詞(檔)
with open('stopwords.txt','r',encoding='utf-8') as f:
    stopwords = f.readlines()
stopwords = [i.strip() for i in stopwords]

#X
#结巴分词并去除停用词
def jieba_stop(each):
    result = []
    jieba_line = jieba.lcut(each)  #jieb.cut()返回迭代器，节省空间；jieba.lcut() 直接返回分词后的列表。三种分词模式：默认精确模式，用于文本分析  
    for l in jieba_line:
        if (l not in stopwords) and (l != ''): # 去除停用词以及空格
            result.append(l.strip())
    return result
xx = train.apply(jieba_stop)  #注意 apply 的用法
# date_news.news = date_news.news.apply(lambda x: ','.join([i for i in x if i !=''])) #反复去除空字符

##分詞(jieba) 目前僅 tain有分詞.test ,dev尚無


In [ ]:
#使用jieba
# import jieba
# for index, row in train.iterrows():
#     # 對 row['text_a'] 進行分詞
#     words = jieba.cut(row['text_a'])
#     # 将分詞结果转换为字符串，并存回 train 變量中
#     train.loc[index, 'text_a'] = ','.join(words)
#============================================================
#使用ckip_client
# import ckip_client
# for index, row in train.iterrows():
#   # 對 row['text_a'] 進行分詞
#   words = ckip_client.segment(row['text_a'])
#   # 将分詞结果转换为字符串，并存回 train 變量中
#   train.loc[index, 'text_a'] = ','.join(words)
#=======================================================
# 直接使用主題模型提取特徵字
# 導入所需模塊
import re
from gensim import corpora, models

# 分類的label
label_list = ['彩票', '財經', '遊戲', '家居', '房產', '星座', '體育', '股票', '時政', '科技', '娛樂', '社會', '教育', '時尚']

# 分類的文本
text_list = train['text_a'].tolist()

# 定義停用詞(original_stop_words)，在城程式的定義-去除標點符號和數字
original_stop_words = ["，", "。", "；", "：", "“", "”", "（", "）", "《", "》", "、", "？", "「", "」", "『", "』", "‘", "’", "...", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
# 定義停用詞(stop_words)，來字文檔的定義
with open("/content/drive/MyDrive/中文新闻文本标题分类Data/stop_words.txt", "r", encoding="utf-8") as f:
  stop_words = f.read().split("\n")
stop_words += original_stop_words
# print(stop_words)
# # 載入停用詞至jieba
# for word in stop_words:
#     jieba.add_word(word)
#斷詞
# texts = [[word for word in jieba.cut(document)] for document in text_list]
texts = [[word for word in jieba.cut(document) if word not in stop_words] for document in text_list]

# 建立詞袋字典
dictionary = corpora.Dictionary(texts)

# 建立詞袋資料集
corpus = [dictionary.doc2bow(text) for text in texts]

# 建立LDA模型
lda = models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=len(label_list))

# 使用下面的方式，指定每個主題的名稱
for i, topic in enumerate(lda.show_topics(num_topics=len(label_list), num_words=10)):
  print('主題{}:{}'.format(label_list[i], topic))

# 註: 上述程式碼中的num_topics參數可以根據你的需求調整，num_words參數可以決定每個主題中顯示的詞語數量。

re.findall
re.findall(r'"(.*?)"', topic[1])
re.findall(pattern, string) 是 Python 內建的正規表達式模組中的一個函數，它會在字符串中尋找所有符合模式的子字符串，並將它們作為一個列表返回。

r'"(.*?)"' 是一個正規表達式模式，它由兩個部分組成：

" 和 "：表示要尋找的子字符串需要由雙引號括起來。
(.*?)：表示要尋找的子字符串需要包含任意字符，並且使用非貪婪模式，因此在找到第一個双引號之後，會在遇到第二個双引號前停止，而不是尋找到整個字符串中的所有双引號。

r'"(.*?)"' : 則是一個正則表達式，表示在字符串中尋找以双引號開頭，以双引號結尾，中間包含任意字符的子字符串。

而這段程式碼中，使用re.findall函數來尋找 topic[1] 中所有符合上述正則表達式的子字符串，並將其作為一個列表返回。

因此在這個程式碼中，將會在topic[1]中找出所有被双引號包含的詞，並將其加入topic_words中。
其他寫法：
#topic_words += re.findall(r'"(\w+)"', topic[1]
#topic_words +=  re.findall(r'\b\w+\b', topic[1])

# 主題模型取出特徵詞

In [ ]:
import re
# 將主題模型選出的特徵字詞放入訓練集中
# 將特徵字詞提取出來
topics = lda.show_topics(num_topics=len(label_list), num_words=100)#這邊設定要的特徵詞數量
# print(topics)
# 定義一個空的 list
topic_words = []

# 對於每個主題
for topic in topics:
  topic_words += re.findall(r'"(.*?)"', topic[1])

# 將特徵字詞加入到訓練集中
# train['text_a_topics'] = train['text_a'].apply(lambda x: " ".join([word for word in topic_words if word in x]))

In [ ]:
# 將 topic_words 轉換為 set，因為set 的查詢效率是 O(1)，比 List 的 O(n) 效率高。
topic_words_set = set(topic_words)


In [ ]:
# 使用 apply 方法，將每一行文本傳入函數中處理
# train['text_a_topics'] = train['text_a'].apply(lambda text: " ".join([word for word in text.split() if word in topic_words_set]))
# # 使用 apply 方法，將每一個
# ==========================
# 另外，在計算時可以考慮使用Pandas.Series.str.count()，它能夠對於一個字符串序列計算出每一個元素中某個子字符串的出現次數，可以減少迴圈的次數。
# 這樣的寫法會將每一行文本中每一個topic_words_set中的詞語的數量加起來。
# train['text_a_topics'] = train['text_a'].str.count('|'.join(topic_words_set))
def count_word(text, words):
    return sum(text.count(word) for word in words)

train['textHavdWordCorrespondToTheTopicsWord'] = train['text_a'].apply(lambda text: count_word(text, topic_words_set))


In [ ]:
# train
train['textHavdWordCorrespondToTheTopicsWord'] 

# 建立df_topicWords用於訓練模型

In [ ]:
import re
# 建立空的 DataFrame
df_topicWords = pd.DataFrame(columns=['topic', 'keywords'])

for topic in topics:
    keywords = re.findall(r'"(.*?)"', topic[1])
    df_topicWords = df_topicWords.append({'topic': topic[0], 'keywords': keywords}, ignore_index=True)
#class0用於二原分類topic0
df_topicWords = df_topicWords.assign(class0=[1,0,0,0,0,0,0,0,0,0,0,0,0,0])
# df_topicWords[class0]=[1,0,0,0,0,0,0,0,0,0,0,0,0,0]
print(df_topicWords)

# 訓練模型

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# 定義訓練資料和標籤
X_train = df_topicWords["keywords"].tolist()
y_train = df_topicWords["class0"].tolist()

# 因為X_train設置為列表，而CountVectorizer需要字符串或文本文件。
# 所以X_train轉換為字符串並使用join函數，然後在轉換為特徵向量之前將其輸入到CountVectorizer中
X_train = [" ".join(words) for words in X_train]
# 將文本轉為特徵向量
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)
print(X_train)
#(0,4) 1 => 0表示文本0號，4表示文本內的一個詞，1表示出現頻率
# 訓練 Logistic Regression 模型
clf = LogisticRegression()
clf.fit(X_train, y_train)

PS
CountVectorizer()也可用停用詞

In [ ]:
X_train

In [ ]:
y_train 

# 評估模型

In [ ]:
X_test=pd.read_table('/content/drive/MyDrive/中文新闻文本标题分类Data/dev.txt', sep='\t', header=None, nrows=20000)
# 添加列名便于对数据进行更好处理
X_test.columns = ["text_a",'label']
print(X_test)
#  train['text_a_topics'] = train['text_a'].apply(lambda text: " ".join([word for word in text.split() if word in topic_words_set]))
X_test['label']=X_test['label'].apply(lambda number:)

In [ ]:
from sklearn.metrics import accuracy_score

# 定義測試資料和標籤
X_test = [新文本]
y_test = [新文本的標籤]

# 預測測試資料
y_pred = clf.predict(X_test)

# 計算準


In [ ]:
# 定義新文本
# new_text = "這是一個新文本"
new_text = "我想買彩票"
# 將新文本轉換成特徵向量
new_text_vect = vectorizer.transform([new_text])

# 使用訓練好的分類器去分類新文本
prediction = clf.predict(new_text_vect)
print(prediction)


In [ ]:
#精度（precision）
from sklearn.metrics import precision_score


y_pred_MNB = clf.predict(X_test)
precision = precision_score(y_test, y_pred_MNB, average='micro') # 计算 micro 召回率
print("precision (micro): ", precision)

precision = precision_score(y_test, y_pred_MNB, average='macro') # 计算 macro 召回率
print("precision (macro): ", precision)

precision = precision_score(y_test, y_pred_MNB, average='weighted') # 计算 weighted 召回率
print("precision (weighted): ", precision)

4. 网格搜索寻最优模型及最优参数

In [ ]:
#跑很久
train_size = 1000
#数据集划分
y = train['label']
X_train = news_tfidf[:train_size,:]  #训练集X
X_test = news_tfidf[train_size:,:]   #测试集X
y_train = y.iloc[:train_size,]  
y_test = y.iloc[train_size:,]  
lr = LogisticRegression()  #模型拟合
MNB = MultinomialNB()
lr_params = {'C':[0.1,0.5,1,10],'random_state':[0]}  #参数
MNB_params = {'alpha':[0.1,0.5,1,10]}

rf = RandomForestRegressor()
rf_params = {'n_estimators':[500,1000],'max_depth':[5,10],'random_state':[0]}

 #网格搜索
lr_grid = GridSearchCV(lr,lr_params,n_jobs=-1,cv=10,verbose=1,scoring='neg_mean_squared_error')
lr_grid.fit(X_train,y_train)
MNB_grid = GridSearchCV(MNB,MNB_params,n_jobs=-1,cv=10,verbose=1,scoring='neg_mean_squared_error')
MNB_grid.fit(X_train,y_train)

rf_grid = GridSearchCV(rf,rf_params,n_jobs=-1,cv=10,verbose=1,scoring='neg_mean_squared_error')
rf_grid.fit(X_train,y_train)

print('LR best params: '+ str(lr_grid.best_params_))  #最优参数
print('LR best score: '+ str(lr_grid.best_score_))        #最优分数

print('RF best params: '+ str(rf_grid.best_params_))
print('RF best score: '+ str(rf_grid.best_score_))

print('MNB best params: '+ str(MNB_grid.best_params_))
print('MNB best score: '+ str(MNB_grid.best_score_))

5.預測

In [ ]:
y_pred = lr_grid.predict(X_test) #使用 LogisticRegression 模型进行预测

y_pred_MNB = MNB_grid.predict(X_test) #使用 MultinomialNB 模型进行预测

#y_pred_rf = rf_grid.predict(X_test) #使用 RandomForestRegressor 模型进行预测

In [ ]:
y_pred


In [ ]:
y_pred = lr_grid.predict("測試分類跑出結果")

6.評估模型

In [ ]:
y_test

In [ ]:
type(y_test)

In [ ]:
type(y_pred_lr)

隨機森林

In [ ]:
# #精度（precision）
# from sklearn.metrics import precision_score
# y_pred_rf = rf_gri.predict(X_test)
# precision = precision_score(y_test, y_pred_rf, average='micro') # 计算 micro 召回率
# print("precision (micro): ", precision)

# precision = precision_score(y_test, y_pred_rf, average='macro') # 计算 macro 召回率
# print("precision (macro): ", precision)

# precision = precision_score(y_test, y_pred_rf, average='weighted') # 计算 weighted 召回率
# print("precision (weighted): ", precision)

MultinomialNB

In [ ]:
#精度（precision）
from sklearn.metrics import precision_score
y_pred_MNB = MNB_grid.predict(X_test)
precision = precision_score(y_test, y_pred_MNB, average='micro') # 计算 micro 召回率
print("precision (micro): ", precision)

precision = precision_score(y_test, y_pred_MNB, average='macro') # 计算 macro 召回率
print("precision (macro): ", precision)

precision = precision_score(y_test, y_pred_MNB, average='weighted') # 计算 weighted 召回率
print("precision (weighted): ", precision)

lr

In [ ]:
#精度（precision）
from sklearn.metrics import precision_score
y_pred_lr = lr_grid.predict(X_test)
precision = precision_score(y_test, y_pred_lr, average='micro') # 计算 micro 召回率
print("precision (micro): ", precision)

precision = precision_score(y_test, y_pred_lr, average='macro') # 计算 macro 召回率
print("precision (macro): ", precision)

precision = precision_score(y_test, y_pred_lr, average='weighted') # 计算 weighted 召回率
print("precision (weighted): ", precision)


In [ ]:
#召回率（recall）
from sklearn.metrics import recall_score

y_pred_lr = lr_grid.predict(X_test)
recall = recall_score(y_test, y_pred_lr, average='micro') # 计算 micro 召回率
print("Recall (micro): ", recall)

recall = recall_score(y_test, y_pred_lr, average='macro') # 计算 macro 召回率
print("Recall (macro): ", recall)

recall = recall_score(y_test, y_pred_lr, average='weighted') # 计算 weighted 召回率
print("Recall (weighted): ", recall)


##模型應用

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 定义新的数值特征
new_data = ['这是一篇关于机器学习的文章', '这是一篇关于人工智能的文章']

# 定义 TfidfVectorizer 实例
tfidf = TfidfVectorizer()

# 使用训练数据训练 TfidfVectorizer
tfidf.fit(train['text_a'])

# 使用训练好的 TfidfVectorizer 转换新的数值特征
new_tfidf = tfidf.transform(new_data)

# 使用训练好的模型 lr_grid 预测新的数值特征
predictions = lr_grid.predict(new_tfidf)

print(predictions)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 定义新的数值特征
new_data = ['''講座內容：
在講座中，王小棣導演將從自身經驗出發，分享豐富的生命故事與人生歷練，並帶出近期拍攝、上映之影集【茁劇場–誰說媽媽像月亮】，分享拍攝過程與內容，以及個人理念。講座第一個小時為 【茁劇場–誰說媽媽像月亮】之影片欣賞。

講師簡介：
王小棣導演於1979年回台工作，擔任多部影片導演與編劇，數度獲得金鐘獎肯定，近期著名作品為 《 45 度 C 天空下》、《 波麗士大人》、《 我在墾丁天氣晴》……等電視劇。
2016 年，王小棣導演結合多名導演，催生【植劇場】， 深耕臺灣戲劇，培植優秀戲劇從業人員。植劇場在第 52 屆電 視金鐘獎取得 24 項入圍紀錄，並奪得金鐘獎戲劇節目、戲劇節目女 主角、戲劇節目女配角……等 5 項大獎，獲得外界極大肯定。
2022年，開啟植劇場的續篇【茁劇場】，持續深耕台灣戲劇界。''']

# 定义 TfidfVectorizer 实例
tfidf = TfidfVectorizer()

# 使用训练数据训练 TfidfVectorizer
tfidf.fit(train['text_a'])

# 使用训练好的 TfidfVectorizer 转换新的数值特征
new_tfidf = tfidf.transform(new_data)

# 使用训练好的模型 lr_grid 预测新的数值特征
predictions = lr_grid.predict(new_tfidf)

print(predictions)

In [ ]:
new_tfidf

##舊版

from sklearn.feature_extraction.text import TfidfVectorizer

# 定义新的数值特征
new_samples = ['這是一條新聞', '這是一個很有意思的事情']

# 使用训练时使用的 TfidfVectorizer 转换为 Tf-idf 矩阵
# vectorizer = TfidfVectorizer(stop_words=stopwords)
vectorizer = TfidfVectorizer()
X_new = vectorizer.fit_transform(new_samples)

# 使用 lr_grid 预测分类结果
y_pred = lr_grid.predict(X_new)

print(y_pred)  # 输出 ['科技', '娱乐']

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
<ipython-input-48-6ac667864f40> in <module>
     10 
     11 # 使用 lr_grid 预测分类结果
---> 12 y_pred = lr_grid.predict(X_new)
     13 
     14 print(y_pred)  # 输出 ['科技', '娱乐']

5 frames
/usr/local/lib/python3.8/dist-packages/sklearn/base.py in _check_n_features(self, X, reset)
    398 
    399         if n_features != self.n_features_in_:
--> 400             raise ValueError(
    401                 f"X has {n_features} features, but {self.__class__.__name__} "
    402                 f"is expecting {self.n_features_in_} features as input."

ValueError: X has 2 features, but LogisticRegression is expecting 45190 features as input.

In [ ]:
X_train = news_tfidf[:train_size,:]
X_train

In [ ]:
MNB_grid.fit(X_train, y_train)
y_predict = MNB_grid.predict(X_test)     #预测值
mat = confusion_matrix(y_test,y_predict) #混淆矩阵


In [ ]:
mat

In [ ]:
df = pd.DataFrame(mat,columns=label_list,index=label_list)
report_MNB = classification_report(y_test,y_predict)

In [ ]:
df

#LDA測試

In [ ]:
from gensim.test.utils import common_texts
from gensim.corpora.dictionary import Dictionary
from gensim import corpora, models

# Create a corpus from a list of texts
common_dictionary = Dictionary(common_texts)
common_corpus = [common_dictionary.doc2bow(text) for text in common_texts]

# Train the model on the corpus.
lda = models.LdaModel(common_corpus, num_topics=10)

In [ ]:
# Create a new corpus, made of previously unseen documents.
other_texts = ["哈哈，這事科技文本測試"
    # ['computer', 'time', 'graph'],
    # ['survey', 'response', 'eps'],
    # ['human', 'system', 'computer']
]
# other_corpus = [common_dictionary.doc2bow(text) for text in other_texts]這個英文用
texts = [[word for word in jieba.cut(document) if word not in stop_words] for document in other_texts]

unseen_doc = other_corpus[0]
vector = lda[unseen_doc]  # get topic probability distribution for a document
vector

In [ ]:
lda.update(other_corpus)
vector = lda[unseen_doc]
vector 

In [ ]:
 other_corpus